# ColliderFM Diagnostics Explorer

This notebook is based on `scripts/plot_diagnostics.py`, but reorganized for interactive exploration.

It is split into stages so you can inspect and tweak the pipeline step by step:

1. load one detailed event and a separate representation sample
2. inspect the raw dataloader output
3. inspect the exact point-view tensors that go into the model
4. inspect augmentations
5. inspect backbone point features, pooled event embeddings, and prototype outputs

## Setup

This cell configures imports, notebook paths, and plotting defaults.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np
import torch
import torch.nn.functional as F

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from collider_fm.diagnostics import (
    compute_pca,
    encode_view,
    load_checkpoint,
    load_events,
    radius,
    sample_indices,
    tensor_summary,
    to_numpy,
    )
from collider_fm.model import create_small_panda_model
from collider_fm.views import augment_point_view, batch_point_views, build_point_view_from_event

plt.style.use('default')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False

## Configuration

Edit the values in this cell to choose data splits, point caps, device, and optional checkpoint loading.

In [ ]:
SEED = 7
DETAIL_SPLIT = 'train[0:1]'
REPRESENTATION_SPLIT = 'train[:10]'
DATASET_TYPE = 'ttbar'
PU_CONFIG = 'pu0'
CACHE_DIR = '/mnt/ceph/users/ewulff/data/hf'
MAX_TRACKER_HITS = 128
MAX_CALO_HITS = 256
POINT_FEATURE_SAMPLE_SIZE = 2000
TOP_K_PROTOTYPES = 8
CHECKPOINT_PATH = None
SAVE_OUTPUTS = False
OUTPUT_DIR = PROJECT_ROOT / 'diagnostics' / 'notebook_explorer'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Using device: {DEVICE}')
print(f'Saving outputs: {SAVE_OUTPUTS}')
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Output directory: {OUTPUT_DIR}')

## Shared Helpers

These notebook-only helpers now sit on top of `collider_fm.diagnostics`, so the interactive explorer and batch script share the same data loading, encoding, and summary logic.

In [ ]:
def maybe_save(fig: plt.Figure, name: str) -> None:
    """Show a figure inline and optionally save the same image to disk."""
    fig.tight_layout()
    if SAVE_OUTPUTS:
        fig.savefig(OUTPUT_DIR / name, dpi=200, bbox_inches='tight')
    plt.show()


def create_model(device: torch.device):
    """Construct the compact diagnostics model and optionally load a checkpoint."""
    model = create_small_panda_model(device=device)
    if CHECKPOINT_PATH is not None:
        checkpoint_artifact = load_checkpoint(model, CHECKPOINT_PATH)
        print('Loaded checkpoint')
        print('Missing keys:', checkpoint_artifact['missing_keys'])
        print('Unexpected keys:', checkpoint_artifact['unexpected_keys'])
    return model

## Load Events

The detailed event drives the geometry and input plots. The representation sample is used for PCA summaries.

In [ ]:
# The detailed event drives all single-event plots, while the separate
# representation sample is reserved for batch-level summaries such as PCA.
detail_events = load_events(
    split=DETAIL_SPLIT,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
    batch_size=64,
 )
representation_events = load_events(
    split=REPRESENTATION_SPLIT,
    dataset_type=DATASET_TYPE,
    pu_config=PU_CONFIG,
    cache_dir=CACHE_DIR,
    batch_size=64,
 )

detail_event = detail_events[0]
print('Detailed event loaded')
print('tracker hits:', len(detail_event['tracker_hits']['x']))
print('calo hits:', len(detail_event['calo_hits']['x']))
print('representation sample size:', len(representation_events))

In [ ]:
print("Number of detail events:", len(detail_events))
detail_event = detail_events[0]
print("Contents of a single detail event:")
for key, value in detail_event.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for subkey, subvalue in value.items():
            if hasattr(subvalue, 'shape'):
                print(f"  {subkey}: shape={subvalue.shape}, dtype={subvalue.dtype}")
            else:
                print(f"  {subkey}: {type(subvalue).__name__} = {subvalue}")
    else:
        print(f"{key}: {value}")

In [ ]:
detail_event['particles']['particle_id']

## Raw Event Diagnostics

These plots show the event exactly as it comes out of the dataloader.

In [ ]:
particles = detail_event.get('particles', {})
particles

In [ ]:
PDG_LABELS = {
    11: 'e-',
    -11: 'e+',
    13: 'mu-',
    -13: 'mu+',
    22: 'gamma',
    111: 'pi0',
    130: 'K0_L',
    211: 'pi+',
    -211: 'pi-',
    2112: 'n',
    -2112: 'nbar',
    2212: 'p',
    -2212: 'pbar',
    321: 'K+',
    -321: 'K-',
}


def pdg_label(pdg_id: int) -> str:
    label = PDG_LABELS.get(int(pdg_id))
    if label is not None:
        return f'{label} ({int(pdg_id)})'
    return f'PDG {int(pdg_id)}'


def tracker_pdg_ids(event: dict[str, Any]) -> np.ndarray:
    tracker_hits = event['tracker_hits']
    particles = event.get('particles', {})

    tracker_particle_ids = to_numpy(tracker_hits['particle_id'])
    particle_ids = to_numpy(particles['particle_id'])
    pdg_ids = to_numpy(particles['pdg_id'])

    particle_to_pdg = {int(particle_id): int(pdg_id) for particle_id, pdg_id in zip(particle_ids, pdg_ids)}
    return np.array([particle_to_pdg[int(particle_id)] for particle_id in tracker_particle_ids])


def plot_raw_geometry(event: dict[str, Any]) -> None:
    tracker_hits = event['tracker_hits']
    calo_hits = event['calo_hits']
    tracker_pdgs = tracker_pdg_ids(event)

    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    tracker_x = to_numpy(tracker_hits['x'])
    tracker_y = to_numpy(tracker_hits['y'])
    tracker_z = to_numpy(tracker_hits['z'])
    unique_pdgs = np.unique(tracker_pdgs)
    tracker_cmap = plt.get_cmap('tab20', max(len(unique_pdgs), 1))

    for index, pdg_id in enumerate(unique_pdgs):
        mask = tracker_pdgs == pdg_id
        label = 'unmatched tracker hit' if int(pdg_id) == 0 else pdg_label(int(pdg_id))
        ax.scatter(
            tracker_z[mask],
            tracker_x[mask],
            tracker_y[mask],
            s=2,
            alpha=0.55,
            label=label,
            color=tracker_cmap(index),
        )

    energy = to_numpy(calo_hits.get('total_energy', torch.zeros_like(calo_hits['x'], dtype=torch.float32)))

    positive_energy = energy[energy > 0]
    if positive_energy.size > 0:
        color_vmin = float(positive_energy.min())
        color_vmax = float(positive_energy.max())
        log_energy = np.log10(np.clip(energy, color_vmin, None))
        p_lo, p_hi = np.percentile(log_energy[energy > 0], [5, 95])
        denom = max(p_hi - p_lo, 1e-6)
        scaled = np.clip((log_energy - p_lo) / denom, 0.0, 1.0)
        marker_size = 3.0 + (scaled ** 2.2) * 110.0
        norm = LogNorm(vmin=color_vmin, vmax=color_vmax)
    else:
        marker_size = np.full_like(energy, 4.0)
        norm = None

    calo = ax.scatter(
        to_numpy(calo_hits['z']),
        to_numpy(calo_hits['x']),
        to_numpy(calo_hits['y']),
        s=marker_size,
        alpha=0.55,
        c=energy,
        cmap='inferno',
        norm=norm,
        label='calo',
    )
    colorbar = fig.colorbar(calo, ax=ax, shrink=0.7, pad=0.1)
    colorbar.set_label('calo energy (log scale)')
    ax.set_xlabel('z [mm]')
    ax.set_ylabel('x [mm]')
    ax.set_zlabel('y [mm]')
    ax.set_title('Raw dataloader event geometry')
    ax.legend(loc='upper right', fontsize=8)
    maybe_save(fig, 'raw_event_geometry.png')


def plot_raw_scalars(event: dict[str, Any]) -> None:
    tracker_hits = event['tracker_hits']
    calo_hits = event['calo_hits']
    tracker_radius = radius(tracker_hits)
    calo_radius = radius(calo_hits)
    tracker_z = to_numpy(tracker_hits['z'])
    calo_z = to_numpy(calo_hits['z'])
    tracker_time = tracker_hits.get('time', torch.zeros(1))
    calo_energy = calo_hits.get('total_energy', torch.zeros(1))

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()
    axes[0].hist(tracker_z, bins=40, color='tab:blue')
    axes[0].set_title('Tracker z')
    axes[1].hist(calo_energy, bins=40,color='tab:red', log=True)
    axes[1].set_title('Calo energy')
    axes[1].set_xlabel('energy (log scale)')
    axes[2].hist(to_numpy(tracker_radius), bins=40, color='tab:cyan')
    axes[2].set_title('Tracker radius')
    axes[3].hist(calo_z, bins=40, color='tab:purple')
    axes[3].set_title('Calo z')
    axes[4].hist(to_numpy(calo_radius), bins=40, color='tab:orange')
    axes[4].set_title('Calo radius')
    axes[5].axis('off')
    axes[5].text(
        0.0,
        0.8,
        '\n'.join([
            f"tracker hits: {len(tracker_hits['x'])}",
            f"calo hits: {len(calo_hits['x'])}",
            f"tracker z range: [{tracker_z.min():.2f}, {tracker_z.max():.2f}]",
            f"calo z range: [{calo_z.min():.2f}, {calo_z.max():.2f}]",
            f"tracker time range: [{tracker_time.min().item():.2f}, {tracker_time.max().item():.2f}]",
            f"calo energy range: [{calo_energy.min().item():.5f}, {calo_energy.max().item():.5f}]",
        ]),
        fontsize=11,
        va='top',
    )
    fig.suptitle('Raw dataloader scalar summaries')
    maybe_save(fig, 'raw_event_scalars.png')


plot_raw_geometry(detail_event)
plot_raw_scalars(detail_event)

## Point-View Construction

These cells show the exact tensors that go into the model after preprocessing and augmentation.

In [ ]:
base_view = build_point_view_from_event(
    detail_event,
    device=DEVICE,
    max_tracker_hits=MAX_TRACKER_HITS,
    max_calo_hits=MAX_CALO_HITS,
 )
aug_view_a = augment_point_view(base_view)
aug_view_b = augment_point_view(base_view)

print('Base view summary:')
print(json.dumps({
    'coord': tensor_summary(base_view['coord']),
    'feat': tensor_summary(base_view['feat']),
    'offset': tensor_summary(base_view['offset'].float()),
}, indent=2))

In [ ]:
base_view.keys()

In [ ]:
base_view['coord'].shape, base_view['feat'].shape, base_view['offset'].shape

In [ ]:
def plot_view_detector_type(view: dict[str, torch.Tensor]) -> None:
    coord = to_numpy(view['coord'])
    detector_type = to_numpy(view['feat'][:, 5])
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')

    tracker_mask = detector_type == 0
    calo_mask = detector_type == 1
    ax.scatter(coord[tracker_mask, 2], coord[tracker_mask, 0], coord[tracker_mask, 1], color='tab:blue', s=4, alpha=0.7, label='tracker (0)')
    ax.scatter(coord[calo_mask, 2], coord[calo_mask, 0], coord[calo_mask, 1], color='tab:red', s=4, alpha=0.7, label='calo (1)')
    ax.set_xlabel('z')
    ax.set_ylabel('x')
    ax.set_zlabel('y')
    ax.set_title('Model input point cloud colored by detector type')
    ax.legend(loc='upper right')
    maybe_save(fig, 'input_detector_type.png')


def plot_view_signal(view: dict[str, torch.Tensor]) -> None:
    coord = to_numpy(view['coord'])
    signal = to_numpy(view['feat'][:, 4])
    features = to_numpy(view['feat'])
    detector_type = features[:, 5].astype(int)
    detector_counts = np.bincount(detector_type, minlength=2)

    fig = plt.figure(figsize=(14, 12))
    grid = fig.add_gridspec(3, 2)
    ax_scatter = fig.add_subplot(grid[:, 0], projection='3d')
    scatter = ax_scatter.scatter(coord[:, 2], coord[:, 0], coord[:, 1], c=signal, cmap='viridis', s=4, alpha=0.7)
    fig.colorbar(scatter, ax=ax_scatter, shrink=0.7, pad=0.1, label='signal channel')
    ax_scatter.set_xlabel('z')
    ax_scatter.set_ylabel('x')
    ax_scatter.set_zlabel('y')
    ax_scatter.set_title('Model input colored by signal')

    axes = [fig.add_subplot(grid[0, 1]), fig.add_subplot(grid[1, 1]), fig.add_subplot(grid[2, 1])]
    axes[0].hist(features[:, 0], bins=40, alpha=0.8, label='x')
    axes[0].hist(features[:, 1], bins=40, alpha=0.6, label='y')
    axes[0].hist(features[:, 2], bins=40, alpha=0.5, label='z')
    axes[0].legend(fontsize=8)
    axes[0].set_title('Coordinate feature histograms')

    axes[1].hist(features[:, 3], bins=40, alpha=0.8, label='radius')
    axes[1].hist(features[:, 4], bins=40, alpha=0.6, label='signal')
    axes[1].legend(fontsize=8)
    axes[1].set_title('Continuous derived feature histograms')

    axes[2].bar([0, 1], detector_counts, color=['tab:blue', 'tab:red'], tick_label=['tracker (0)', 'calo (1)'])
    axes[2].set_title('Detector type counts')
    axes[2].set_ylabel('points')
    maybe_save(fig, 'input_signal_and_features.png')


def plot_augmentations(base_view: dict[str, torch.Tensor], aug_a: dict[str, torch.Tensor], aug_b: dict[str, torch.Tensor]) -> None:
    views = [('base', base_view), ('aug A', aug_a), ('aug B', aug_b)]
    fig = plt.figure(figsize=(18, 6))
    for index, (label, view) in enumerate(views, start=1):
        coord = to_numpy(view['coord'])
        detector_type = to_numpy(view['feat'][:, 5])
        ax = fig.add_subplot(1, 3, index, projection='3d')
        ax.scatter(coord[:, 2], coord[:, 0], coord[:, 1], c=detector_type, cmap='coolwarm', s=4, alpha=0.7)
        ax.set_title(label)
        ax.set_xlabel('z')
        ax.set_ylabel('x')
        ax.set_zlabel('y')
    fig.suptitle('Detailed event: base and augmented views')
    maybe_save(fig, 'augmentations.png')


def plot_augmentation_delta(base_view: dict[str, torch.Tensor], aug_a: dict[str, torch.Tensor], aug_b: dict[str, torch.Tensor]) -> None:
    base_coord = base_view['coord']
    base_signal = base_view['feat'][:, 4]
    deltas = {
        'aug A': {
            'coord': torch.linalg.norm(aug_a['coord'] - base_coord, dim=1),
            'signal': aug_a['feat'][:, 4] - base_signal,
        },
        'aug B': {
            'coord': torch.linalg.norm(aug_b['coord'] - base_coord, dim=1),
            'signal': aug_b['feat'][:, 4] - base_signal,
        },
    }
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label, values in deltas.items():
        axes[0].hist(to_numpy(values['coord']), bins=40, alpha=0.6, label=label)
        axes[1].hist(to_numpy(values['signal']), bins=40, alpha=0.6, label=label)
    axes[0].set_title('Coordinate displacement magnitude')
    axes[1].set_title('Signal-channel perturbation')
    for axis in axes:
        axis.legend()
    fig.suptitle('Augmentation deltas relative to the base view')
    maybe_save(fig, 'augmentation_delta.png')


plot_view_detector_type(base_view)
plot_view_signal(base_view)
plot_augmentations(base_view, aug_view_a, aug_view_b)
plot_augmentation_delta(base_view, aug_view_a, aug_view_b)

## Model Diagnostics

This section extracts the learned representations from the student and teacher networks. The representation plots require CUDA with the current PTv3/spconv stack.

In [ ]:
def plot_logits(student_probs: list[np.ndarray], teacher_probs: list[np.ndarray], top_k: int) -> None:
    stacked = np.vstack(student_probs + teacher_probs)
    mean_scores = stacked.mean(axis=0)
    top_indices = np.argsort(mean_scores)[-top_k:]
    labels = [str(index) for index in top_indices]
    x = np.arange(len(top_indices))
    width = 0.2

    fig, ax = plt.subplots(figsize=(12, 5))
    series = [
        ('student A', student_probs[0][top_indices]),
        ('student B', student_probs[1][top_indices]),
        ('teacher A', teacher_probs[0][top_indices]),
        ('teacher B', teacher_probs[1][top_indices]),
    ]
    for index, (label, values) in enumerate(series):
        ax.bar(x + (index - 1.5) * width, values, width=width, label=label)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel('prototype index')
    ax.set_ylabel('probability')
    ax.set_title('Detailed event: top prototype distributions')
    ax.legend()
    maybe_save(fig, 'prototype_logits.png')


def jensen_shannon_divergence(prob_a: torch.Tensor, prob_b: torch.Tensor) -> float:
    mean_prob = 0.5 * (prob_a + prob_b)
    js = 0.5 * F.kl_div(prob_a.log(), mean_prob, reduction='sum') + 0.5 * F.kl_div(prob_b.log(), mean_prob, reduction='sum')
    return float(js.item())


def embedding_cosine_similarity(embedding_a: torch.Tensor, embedding_b: torch.Tensor) -> float:
    return float(F.cosine_similarity(embedding_a.reshape(1, -1), embedding_b.reshape(1, -1), dim=1).item())


def plot_view_agreement(base_pooled: torch.Tensor, aug_a_pooled: torch.Tensor, aug_b_pooled: torch.Tensor, student_probs: list[torch.Tensor], teacher_probs: list[torch.Tensor]) -> None:
    cosine_values = {
        'base vs aug A': embedding_cosine_similarity(base_pooled, aug_a_pooled),
        'base vs aug B': embedding_cosine_similarity(base_pooled, aug_b_pooled),
        'aug A vs aug B': embedding_cosine_similarity(aug_a_pooled, aug_b_pooled),
    }
    js_values = {
        'student': jensen_shannon_divergence(student_probs[0], student_probs[1]),
        'teacher': jensen_shannon_divergence(teacher_probs[0], teacher_probs[1]),
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(list(cosine_values.keys()), list(cosine_values.values()), color='tab:green')
    axes[0].set_ylim(0.0, 1.05)
    axes[0].set_title('Embedding cosine similarity')
    axes[0].tick_params(axis='x', rotation=20)
    axes[1].bar(list(js_values.keys()), list(js_values.values()), color='tab:purple')
    axes[1].set_title('Jensen-Shannon divergence')
    fig.suptitle('Detailed event: view agreement')
    maybe_save(fig, 'view_agreement.png')


def plot_embedding_pca(embeddings: torch.Tensor) -> None:
    projected = compute_pca(to_numpy(embeddings), n_components=2)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(projected[:, 0], projected[:, 1], c=np.arange(projected.shape[0]), cmap='tab10', s=60)
    for index in range(projected.shape[0]):
        ax.annotate(f'event_{index:03d}', (projected[index, 0], projected[index, 1]), fontsize=8)
    ax.set_title('Pooled event embedding PCA')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    maybe_save(fig, 'embedding_pca.png')


def plot_point_feature_pca(point_features: torch.Tensor, detector_type: torch.Tensor, max_points: int, seed: int) -> None:
    point_features_np = to_numpy(point_features)
    detector_type_np = to_numpy(detector_type)
    selected = sample_indices(point_features_np.shape[0], max_points, seed)
    projected = compute_pca(point_features_np[selected], n_components=2)

    fig, ax = plt.subplots(figsize=(8, 6))
    tracker_mask = detector_type_np[selected] == 0
    calo_mask = detector_type_np[selected] == 1
    ax.scatter(projected[tracker_mask, 0], projected[tracker_mask, 1], color='tab:blue', s=8, alpha=0.6, label='tracker (0)')
    ax.scatter(projected[calo_mask, 0], projected[calo_mask, 1], color='tab:red', s=8, alpha=0.6, label='calo (1)')
    ax.set_title('Backbone point-feature PCA')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.legend()
    maybe_save(fig, 'point_feature_pca.png')


if DEVICE.type != 'cuda':
    print('CUDA is required for the model-backed diagnostics with the current PTv3/spconv stack.')
else:
    model = create_model(DEVICE)
    model.eval()
    print('Model parameters:', sum(parameter.numel() for parameter in model.parameters()) / 1e6, 'M')

    detail_base_encoding = encode_view(model, base_view, use_teacher=False)
    detail_aug_a_student = encode_view(model, aug_view_a, use_teacher=False)
    detail_aug_b_student = encode_view(model, aug_view_b, use_teacher=False)
    detail_aug_a_teacher = encode_view(model, aug_view_a, use_teacher=True)
    detail_aug_b_teacher = encode_view(model, aug_view_b, use_teacher=True)

    student_probs = [F.softmax(detail_aug_a_student['logits'][0], dim=-1), F.softmax(detail_aug_b_student['logits'][0], dim=-1)]
    teacher_probs = [F.softmax(detail_aug_a_teacher['logits'][0], dim=-1), F.softmax(detail_aug_b_teacher['logits'][0], dim=-1)]

    plot_logits([to_numpy(prob) for prob in student_probs], [to_numpy(prob) for prob in teacher_probs], TOP_K_PROTOTYPES)
    plot_view_agreement(
        detail_base_encoding['pooled'][0],
        detail_aug_a_student['pooled'][0],
        detail_aug_b_student['pooled'][0],
        student_probs,
        teacher_probs,
    )

    representation_views = [
        build_point_view_from_event(
            event,
            device=DEVICE,
            max_tracker_hits=MAX_TRACKER_HITS,
            max_calo_hits=MAX_CALO_HITS,
        )
        for event in representation_events
    ]
    representation_batch = batch_point_views(representation_views)
    representation_encoding = encode_view(model, representation_batch, use_teacher=False)

    plot_embedding_pca(representation_encoding['pooled'])
    plot_point_feature_pca(
        representation_encoding['point_features'],
        representation_batch['feat'][:, 5],
        POINT_FEATURE_SAMPLE_SIZE,
        SEED,
    )

    print('Detailed pooled embedding summary:')
    print(json.dumps(tensor_summary(detail_base_encoding['pooled']), indent=2))

## Optional Artifact Export

If `SAVE_OUTPUTS = True`, this cell writes the same lightweight artifact summaries as the script.

In [ ]:
if SAVE_OUTPUTS:
    artifact_dir = OUTPUT_DIR / 'artifacts'
    artifact_dir.mkdir(parents=True, exist_ok=True)

    payload = {
        'detail_split': DETAIL_SPLIT,
        'representation_split': REPRESENTATION_SPLIT,
        'dataset_type': DATASET_TYPE,
        'pu_config': PU_CONFIG,
        'device': str(DEVICE),
        'checkpoint_path': CHECKPOINT_PATH,
        'max_tracker_hits': MAX_TRACKER_HITS,
        'max_calo_hits': MAX_CALO_HITS,
        'point_feature_sample_size': POINT_FEATURE_SAMPLE_SIZE,
        'seed': SEED,
        'detail_event': {
            'tracker_hits': int(len(detail_event['tracker_hits']['x'])),
            'calo_hits': int(len(detail_event['calo_hits']['x'])),
            'base_view_coord': tensor_summary(base_view['coord']),
            'base_view_feat': tensor_summary(base_view['feat']),
        },
    }
    (artifact_dir / 'notebook_summary.json').write_text(json.dumps(payload, indent=2, sort_keys=True))
    print(f'Wrote {artifact_dir / "notebook_summary.json"}')
else:
    print('Not saving outputs to disk. To enable saving, set SAVE_OUTPUTS = True in the config cell.')